# PSPL Data Engineering Portfolio — Bronze → Silver Transformation

This notebook demonstrates the full PySpark Bronze-to-Silver pipeline:
- Delta Lake reads and writes
- Generic cleaning (deduplication, null-coercion, type casting)
- Window functions
- Time-travel queries
- Matplotlib visualisations

**Requirements covered**: 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 7.2

In [ ]:
# Section 0 — Setup
# Initialize SparkSession with Delta Lake Maven package and configure extensions.

import os
import sys

import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for nbconvert execution
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    DateType, TimestampType, BooleanType,
    IntegerType, LongType, DoubleType, StringType,
)

DELTA_PACKAGE = "io.delta:delta-spark_2.12:3.2.0"

# Resolve paths relative to the notebook location so the notebook works
# whether executed from the repo root or the notebooks/ directory.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
REPO_ROOT = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == "notebooks" else NOTEBOOK_DIR
DELTA_DIR = os.path.join(REPO_ROOT, "delta_lake")
DOCS_DIR  = os.path.join(REPO_ROOT, "docs", "sample_outputs")
os.makedirs(DOCS_DIR, exist_ok=True)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from ingest.spark_preflight import ensure_pyspark_uses_current_interpreter  # noqa: E402
from ingest.spark_runtime_paths import configure_spark_local_dirs, stop_spark_quietly  # noqa: E402

ensure_pyspark_uses_current_interpreter()

_builder = (
    SparkSession.builder
    .master("local[2]")
    .appName("PSPL-delta-lake-operations")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.jars.packages", DELTA_PACKAGE)
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_local_dirs(_builder, REPO_ROOT).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Delta dir     : {DELTA_DIR}")
print(f"Docs dir      : {DOCS_DIR}")

In [ ]:
# Section 1 — Bronze reads
# Load all 9 Bronze Delta tables into named PySpark DataFrames.

def bronze_path(name: str) -> str:
    return os.path.join(DELTA_DIR, "bronze", name)

bronze_beneficiaries    = spark.read.format("delta").load(bronze_path("beneficiaries"))
bronze_payments         = spark.read.format("delta").load(bronze_path("payments"))
bronze_surveys          = spark.read.format("delta").load(bronze_path("surveys"))
bronze_inventory        = spark.read.format("delta").load(bronze_path("inventory"))
bronze_complaints       = spark.read.format("delta").load(bronze_path("complaints"))
bronze_donor_reports    = spark.read.format("delta").load(bronze_path("donor_reports"))
bronze_afghan_refugees  = spark.read.format("delta").load(bronze_path("afghan_refugees"))
bronze_refugee_assistance  = spark.read.format("delta").load(bronze_path("refugee_assistance"))
bronze_refugee_protection  = spark.read.format("delta").load(bronze_path("refugee_protection"))

BRONZE_TABLES = {
    "beneficiaries":    bronze_beneficiaries,
    "payments":         bronze_payments,
    "surveys":          bronze_surveys,
    "inventory":        bronze_inventory,
    "complaints":       bronze_complaints,
    "donor_reports":    bronze_donor_reports,
    "afghan_refugees":  bronze_afghan_refugees,
    "refugee_assistance":  bronze_refugee_assistance,
    "refugee_protection":  bronze_refugee_protection,
}

print("Bronze tables loaded:", list(BRONZE_TABLES.keys()))

In [ ]:
# Section 2 — Row count summary
# Display a pandas DataFrame showing Bronze row counts for all 9 tables.

bronze_counts = {name: sdf.count() for name, sdf in BRONZE_TABLES.items()}

bronze_summary = pd.DataFrame(
    list(bronze_counts.items()),
    columns=["table", "bronze_row_count"],
).set_index("table")

print("=== Bronze Row Counts ===")
print(bronze_summary.to_string())

In [ ]:
# Section 3 — Cleaning function
# Import clean_dataframe from ingest/transforms.py.
# The function:
#   (a) drops fully duplicate rows via .dropDuplicates()
#   (b) replaces empty strings with null for all STRING columns
#   (c) casts columns to correct Silver types by name pattern

# Ensure the repo root is on sys.path so the ingest package is importable.
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from ingest.transforms import clean_dataframe  # noqa: E402

print("clean_dataframe imported from ingest.transforms")
help(clean_dataframe)

In [ ]:
# Section 4 — Silver writes
# Apply clean_dataframe() to each Bronze table and write to delta_lake/silver/{name}/.

def silver_path(name: str) -> str:
    return os.path.join(DELTA_DIR, "silver", name)

silver_tables = {}

for name, bronze_sdf in BRONZE_TABLES.items():
    print(f"Cleaning and writing silver/{name} …", end=" ")
    cleaned = clean_dataframe(bronze_sdf)
    (
        cleaned.write
        .format("delta")
        .mode("overwrite")
        .save(silver_path(name))
    )
    silver_tables[name] = cleaned
    print("done")

print("\nAll Silver tables written.")

In [ ]:
# Section 5 — Silver row counts
# Display side-by-side Bronze vs Silver row counts and deduplication delta.

silver_counts = {}
for name in BRONZE_TABLES:
    silver_counts[name] = spark.read.format("delta").load(silver_path(name)).count()

comparison = pd.DataFrame({
    "bronze_rows": bronze_counts,
    "silver_rows": silver_counts,
})
comparison["dedup_delta"] = comparison["bronze_rows"] - comparison["silver_rows"]
comparison["dedup_pct"] = (
    comparison["dedup_delta"] / comparison["bronze_rows"] * 100
).round(2)

print("=== Bronze vs Silver Row Counts ===")
print(comparison.to_string())

In [ ]:
# Section 6 — Window function
# On silver/payments, apply row_number() over
# Window.partitionBy("beneficiary_id").orderBy(col("payment_date").desc())
# to add a payment_rank column, then display the top 10 rows.

silver_payments = spark.read.format("delta").load(silver_path("payments"))

payment_window = (
    Window
    .partitionBy("beneficiary_id")
    .orderBy(F.col("payment_date").desc())
)

payments_ranked = silver_payments.withColumn(
    "payment_rank",
    F.row_number().over(payment_window),
)

print("=== Top 10 Rows — Payments with payment_rank ===")
payments_ranked.select(
    "payment_id", "beneficiary_id", "payment_date", "amount", "payment_rank"
).show(10, truncate=False)

In [ ]:
# Section 7 — Time-travel
# Read Bronze beneficiaries at version 0 using .option("versionAsOf", 0).
# Display schema and row count.

bronze_beneficiaries_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(bronze_path("beneficiaries"))
)

print("=== Bronze beneficiaries — version 0 ===")
print(f"Row count : {bronze_beneficiaries_v0.count()}")
print("Schema:")
bronze_beneficiaries_v0.printSchema()

In [ ]:
# Section 8 — Chart 1
# Matplotlib histogram of `amount` from silver/payments.
# Title: "Payment Amount Distribution (PKR)"
# Save to docs/sample_outputs/payment_amount_distribution.png

payments_pd = (
    spark.read.format("delta")
    .load(silver_path("payments"))
    .select("amount")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(
    payments_pd["amount"].dropna(),
    bins=50,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
)
ax.set_title("Payment Amount Distribution (PKR)", fontsize=14, fontweight="bold")
ax.set_xlabel("Amount (PKR)", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

chart1_path = os.path.join(DOCS_DIR, "payment_amount_distribution.png")
fig.savefig(chart1_path, dpi=150)
plt.close(fig)
print(f"Chart saved → {chart1_path}")

In [ ]:
# Section 9 — Chart 2
# Matplotlib histogram of `vulnerability_score` from silver/afghan_refugees
# grouped by `arrival_wave`.
# Title: "Refugee Vulnerability Score by Arrival Wave"
# Save to docs/sample_outputs/vulnerability_histogram.png

refugees_pd = (
    spark.read.format("delta")
    .load(silver_path("afghan_refugees"))
    .select("vulnerability_score", "arrival_wave")
    .toPandas()
)

waves = sorted(refugees_pd["arrival_wave"].dropna().unique())
colors = plt.cm.tab10.colors  # up to 10 distinct colours

fig, ax = plt.subplots(figsize=(11, 6))
for i, wave in enumerate(waves):
    subset = refugees_pd.loc[
        refugees_pd["arrival_wave"] == wave, "vulnerability_score"
    ].dropna()
    ax.hist(
        subset,
        bins=30,
        alpha=0.6,
        label=str(wave),
        color=colors[i % len(colors)],
        edgecolor="white",
        linewidth=0.4,
    )

ax.set_title("Refugee Vulnerability Score by Arrival Wave", fontsize=14, fontweight="bold")
ax.set_xlabel("Vulnerability Score (0–10)", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.legend(title="Arrival Wave", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

chart2_path = os.path.join(DOCS_DIR, "vulnerability_histogram.png")
fig.savefig(chart2_path, dpi=150)
plt.close(fig)
print(f"Chart saved → {chart2_path}")

# Stop SparkSession cleanly at the end of the notebook.
# Uses repo .spark_scratch + clearCache + delay to reduce Windows "Failed to delete" on temp JARs.
stop_spark_quietly(spark)
print("SparkSession stopped.")